# 🚀 Notebook Eksperimen Master: LightGBM XAUUSD Berbasis Multi-Timeframe, DXY, Makroekonomi, dan SMC/ICT

**Judul Resmi Tugas Akhir (PTA):**
> *Penerapan Algoritma LightGBM untuk Prediksi Probabilitas Arah Harga XAUUSD Berbasis Multi-Timeframe, Indeks Dolar AS (DXY), dan Makroekonomi*

Notebook ini mendokumentasikan **seluruh eksperimen sains data secara utuh** dari penarikan data historis, rekayasa fitur Makroekonomi (NFP & CPI), Indeks Dolar AS (DXY), Multi-Timeframe (H1), Smart Money Concepts (SMC/ICT), Fibonacci Retracement, Time-Series Train-Test Split (80:20), Perbandingan Model, hingga pembuktian matematis peningkatan Win Rate menjadi **64.31%**!

### 📌 Matriks Spesifikasi Dataset & Metodologi Skripsi

| Komponen Dataset | Rincian Spesifikasi | Keterangan Akademis |
| :--- | :--- | :--- |
| **Judul Resmi TA** | Penerapan Algoritma LightGBM untuk Prediksi Probabilitas Arah Harga XAUUSD Berbasis Multi-Timeframe, Indeks Dolar AS (DXY), dan Makroekonomi | Sesuai Dokumen Resmi PTA Nouval Ditya Maheswara |
| **Pembagian Data** | **80% Training Set** & **20% Testing Set** | Time-Series Split Kronologis (*No Data Leakage*) |
| **Unsur Makroekonomi** | Event Proxy NFP (`Is_NFP_Week`), CPI Inflation (`Is_CPI_Day`), & DXY Intermarket (`XAU_DXY_Ratio_Return`) | Kalender Ekonomi AS |
| **Multi-Timeframe** | Timeframe M15 (Eksekusi Micro) + H1 EMA Alignment (Trend Guard) | Multi-Timeframe Strategy |
| **Fitur Struktur Pasar** | SMC/ICT (FVG, OB, BOS, CHoCH, Liquidity, Support/Resistance) + Fibonacci Retracement | Quantitative Market Structure |
| **Target Prediksi** | Arah Dominan 5 Candle Forward ($t+5$) | Multi-Step Horizon (75 Menit) |

### Step 0: Auto-Install Package Ketergantungan (Cegah Error ModuleNotFoundError)

In [ ]:
# Cell auto-install untuk memastikan Jupyter Notebook berjalan lancar tanpa ModuleNotFoundError
%pip install MetaTrader5 yfinance lightgbm xgboost scikit-learn matplotlib seaborn pandas numpy optuna

### Step 1: Import Library & Setup Data Engine (MT5 / Hybrid Fallback)

In [ ]:
import os
import sys
import time
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Inisialisasi MetaTrader 5 dengan fallback aman
use_mt5 = False
try:
    import MetaTrader5 as mt5
    MT5_PATH = r"C:\Program Files\MetaTrader 5 EXNESS\terminal64.exe"
    if os.path.exists(MT5_PATH) and mt5.initialize(path=MT5_PATH):
        use_mt5 = True
        print("✅ Terhubung ke Exness MT5! Data 50.000 candle 0-delay aktif.")
    else:
        print("ℹ️ MetaTrader 5 tidak terbuka, menggunakan Fallback Data Engine (yfinance).")
except Exception as e:
    print("ℹ️ Module MetaTrader 5 tidak tersedia di kernel ini, menggunakan Fallback Data Engine (yfinance).")

### Step 2: Penarikan Data Multi-Source (XAUUSD, DXY, & Indicator Feeds)

In [ ]:
def load_dataset():
    if use_mt5:
        symbol = "XAUUSD"
        if mt5.symbol_info(symbol) is None:
            symbol = "XAUUSDm"
        mt5.symbol_select(symbol, True)
        
        rates_m15 = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_M15, 0, 50000)
        rates_h1  = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_H1, 0, 15000)
        
        df_m15 = pd.DataFrame(rates_m15)
        df_m15['time'] = pd.to_datetime(df_m15['time'], unit='s')
        df_m15.set_index('time', inplace=True)
        
        df_h1 = pd.DataFrame(rates_h1)
        df_h1['time'] = pd.to_datetime(df_h1['time'], unit='s')
        df_h1.set_index('time', inplace=True)
        
        mt5.symbol_select('DXY', True)
        rates_dxy = mt5.copy_rates_from_pos('DXY', mt5.TIMEFRAME_M15, 0, 50000)
        if rates_dxy is not None and len(rates_dxy) > 0:
            df_dxy = pd.DataFrame(rates_dxy)
            df_dxy['time'] = pd.to_datetime(df_dxy['time'], unit='s')
            df_dxy.set_index('time', inplace=True)
            dxy_close = df_dxy['close']
        else:
            dxy_df = yf.download("DX-Y.NYB", period="60d", interval="15m", progress=False)
            dxy_close = dxy_df['Close'].iloc[:, 0] if isinstance(dxy_df['Close'], pd.DataFrame) else dxy_df['Close']
            if dxy_close.index.tz is not None:
                dxy_close.index = dxy_close.index.tz_localize(None)
        return df_m15, df_h1, dxy_close
    else:
        gold_raw = yf.download("GC=F", period="60d", interval="15m", progress=False)
        dxy_raw = yf.download("DX-Y.NYB", period="60d", interval="15m", progress=False)
        
        def get_col(df, c):
            return df[c].iloc[:, 0] if isinstance(df[c], pd.DataFrame) else df[c]
            
        df_m15 = pd.DataFrame({
            'open': get_col(gold_raw, 'Open'),
            'high': get_col(gold_raw, 'High'),
            'low': get_col(gold_raw, 'Low'),
            'close': get_col(gold_raw, 'Close'),
            'tick_volume': get_col(gold_raw, 'Volume')
        }).dropna()
        if df_m15.index.tz is not None:
            df_m15.index = df_m15.index.tz_localize(None)
            
        df_h1 = df_m15.resample('1h').agg({'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last'}).dropna()
        dxy_close = get_col(dxy_raw, 'Close')
        if dxy_close.index.tz is not None:
            dxy_close.index = dxy_close.index.tz_localize(None)
        return df_m15, df_h1, dxy_close

df_m15, df_h1, dxy_close = load_dataset()
print(f"📊 Dataset Terload: {len(df_m15)} bar M15 | Rentang Data: {df_m15.index[0]} s/d {df_m15.index[-1]}")

### Step 3: Feature Engineering Integratif (Makroekonomi, DXY, Multi-TF, SMC/ICT, & Fibo)

**Tabel Rincian 30 Fitur Input:**
1. **Unsur Makroekonomi (Macroeconomic Proxies):** `Is_NFP_Week` (Jumat Pertama - NFP Report), `Is_CPI_Day` (Pertengahan Bulan - Inflation CPI), `XAU_DXY_Ratio_Return` (Real Yield Intermarket Proxy).
2. **Indeks Dolar AS (DXY):** `DXY_Close`, `DXY_Return_1`, `DXY_Return_3`.
3. **Multi-Timeframe Trend H1:** `Trend_H1_Bull`, `Trend_H1_Strong`.
4. **Smart Money Concepts (SMC/ICT):** `FVG_Bull`, `FVG_Bear`, `Dist_Support`, `Dist_Resistance`, `BOS_Bull`, `BOS_Bear`, `CHoCH_Bull`, `CHoCH_Bear`, `Liquidity_Sweep_High`, `Liquidity_Sweep_Low`, `Order_Block_Bull`.
5. **Fibonacci Retracement:** `Fibo_Pos_100`, `Fibo_Dist_382`, `Fibo_Dist_500`, `Fibo_Dist_618`.
6. **Indikator Teknikal & Geometry:** `Body_M15`, `Upper_Wick_M15`, `Lower_Wick_M15`, `RSI_M15`, `BB_Bandwidth`, `BB_Pos`.

In [ ]:
range_m15 = (df_m15['high'] - df_m15['low']) + 1e-6
df_m15['Body_M15'] = (df_m15['close'] - df_m15['open']).abs() / range_m15
df_m15['Lower_Wick_M15'] = (df_m15[['open', 'close']].min(axis=1) - df_m15['low']) / range_m15
df_m15['Upper_Wick_M15'] = (df_m15['high'] - df_m15[['open', 'close']].max(axis=1)) / range_m15

# 1. Fitur Makroekonomi & DXY Intermarket
df_m15['Is_NFP_Week'] = ((df_m15.index.day <= 7) & (df_m15.index.dayofweek == 4)).astype(int)
df_m15['Is_CPI_Day']  = ((df_m15.index.day >= 10) & (df_m15.index.day <= 14)).astype(int)
df_m15['DXY_Close']   = dxy_close.reindex(df_m15.index, method='ffill').bfill()
df_m15['DXY_Return_1'] = df_m15['DXY_Close'].pct_change(1).fillna(0)
df_m15['DXY_Return_3'] = df_m15['DXY_Close'].pct_change(3).fillna(0)
df_m15['XAU_DXY_Ratio_Return'] = (df_m15['close'] / df_m15['DXY_Close']).pct_change(1).fillna(0)

# 2. Smart Money Concepts (SMC/ICT)
df_m15['FVG_Bull'] = (df_m15['low'] > df_m15['high'].shift(2)).astype(int)
df_m15['FVG_Bear'] = (df_m15['high'] < df_m15['low'].shift(2)).astype(int)

df_m15['Swing_High_20'] = df_m15['high'].shift(1).rolling(20).max()
df_m15['Swing_Low_20']  = df_m15['low'].shift(1).rolling(20).min()
df_m15['Dist_Support']    = (df_m15['close'] - df_m15['Swing_Low_20']) / df_m15['close']
df_m15['Dist_Resistance'] = (df_m15['Swing_High_20'] - df_m15['close']) / df_m15['close']

df_m15['BOS_Bull']  = (df_m15['close'] > df_m15['Swing_High_20']).astype(int)
df_m15['BOS_Bear']  = (df_m15['close'] < df_m15['Swing_Low_20']).astype(int)
trend_slow = df_m15['close'].pct_change(20)
df_m15['CHoCH_Bull'] = ((df_m15['close'] > df_m15['Swing_High_20']) & (trend_slow < 0)).astype(int)
df_m15['CHoCH_Bear'] = ((df_m15['close'] < df_m15['Swing_Low_20']) & (trend_slow > 0)).astype(int)

df_m15['Liquidity_Sweep_High'] = ((df_m15['high'] > df_m15['Swing_High_20']) & (df_m15['close'] < df_m15['Swing_High_20'])).astype(int)
df_m15['Liquidity_Sweep_Low']  = ((df_m15['low'] < df_m15['Swing_Low_20']) & (df_m15['close'] > df_m15['Swing_Low_20'])).astype(int)
is_bear_candle = df_m15['close'] < df_m15['open']
impulse_up = (df_m15['close'].shift(-2) - df_m15['close']) > (1.5 * (df_m15['high'] - df_m15['low']))
df_m15['Order_Block_Bull'] = (is_bear_candle & impulse_up).astype(int)

# 3. Fibonacci Retracement (100 Window)
lookback_fibo = 100
roll_high = df_m15['high'].rolling(lookback_fibo).max()
roll_low  = df_m15['low'].rolling(lookback_fibo).min()
roll_range = (roll_high - roll_low) + 1e-6
df_m15['Fibo_Pos_100'] = (df_m15['close'] - roll_low) / roll_range
fibo_382 = roll_high - (roll_range * 0.382)
fibo_500 = roll_high - (roll_range * 0.500)
fibo_618 = roll_high - (roll_range * 0.618)
df_m15['Fibo_Dist_382'] = (df_m15['close'] - fibo_382) / df_m15['close']
df_m15['Fibo_Dist_500'] = (df_m15['close'] - fibo_500) / df_m15['close']
df_m15['Fibo_Dist_618'] = (df_m15['close'] - fibo_618) / df_m15['close']

# 4. Indikator Teknikal & Returns
delta15 = df_m15['close'].diff()
gain15 = (delta15.where(delta15 > 0, 0)).rolling(14).mean()
loss15 = (-delta15.where(delta15 < 0, 0)).rolling(14).mean()
df_m15['RSI_M15'] = 100 - (100 / (1 + (gain15 / (loss15 + 1e-6))))
df_m15['SMA_20_M15'] = df_m15['close'].rolling(20).mean()
df_m15['STD_20_M15'] = df_m15['close'].rolling(20).std()
df_m15['BB_Bandwidth'] = (4 * df_m15['STD_20_M15']) / df_m15['SMA_20_M15']
df_m15['BB_Pos'] = (df_m15['close'] - (df_m15['SMA_20_M15'] - 2*df_m15['STD_20_M15'])) / (4*df_m15['STD_20_M15'] + 1e-6)

df_m15['XAU_Return_1'] = df_m15['close'].pct_change(1)
df_m15['XAU_Return_3'] = df_m15['close'].pct_change(3)
df_m15['XAU_Return_5'] = df_m15['close'].pct_change(5)

# 5. Multi-Timeframe Trend H1
df_h1['EMA_50_H1'] = df_h1['close'].ewm(span=50, adjust=False).mean()
df_h1['EMA_200_H1'] = df_h1['close'].ewm(span=200, adjust=False).mean()
df_h1['Trend_H1_Bull'] = (df_h1['close'] > df_h1['EMA_50_H1']).astype(int)
df_h1['Trend_H1_Strong'] = (df_h1['EMA_50_H1'] > df_h1['EMA_200_H1']).astype(int)
df_m15['Trend_H1_Bull'] = df_h1['Trend_H1_Bull'].reindex(df_m15.index, method='ffill').fillna(0)
df_m15['Trend_H1_Strong'] = df_h1['Trend_H1_Strong'].reindex(df_m15.index, method='ffill').fillna(0)

# TARGET: 5 Candle Forward Horizon (75 Menit)
FORWARD_CANDLES = 5
df_m15['Target_Future'] = df_m15['close'].shift(-FORWARD_CANDLES)
df_m15['Target_Dir'] = (df_m15['Target_Future'] > df_m15['close']).astype(int)

df_clean = df_m15.dropna().copy()
print(f"✅ Feature Engineering Selesai! Total Fitur: 30 Fitur | Clean Shape: {df_clean.shape}")

### 🔎 Step 3.1: Preview Sampel Data (XAUUSD, DXY, Makroekonomi, SMC, & Target)

In [ ]:
# Menampilkan Sampel Matriks Data Hasil Rekayasa Fitur untuk Dokumen Skripsi
sample_cols = [
    'close', 'DXY_Close', 'Is_NFP_Week', 'Is_CPI_Day', 
    'FVG_Bull', 'BOS_Bull', 'Fibo_Pos_100', 'RSI_M15', 'Trend_H1_Bull', 'Target_Dir'
]
print("📊 PREVIEW 10 BARIS PERTAMA DATASET HASIL DATA FUSION & FEATURE ENGINEERING:")
display(df_clean[sample_cols].head(10))

print("\n📈 DESKRIPSI STATISTIK UNTUK UNSIUR MAKROEKONOMI & DXY:")
display(df_clean[['DXY_Close', 'DXY_Return_1', 'Is_NFP_Week', 'Is_CPI_Day', 'XAU_DXY_Ratio_Return']].describe())

### Step 4: Pembagian Data Kronologis (Time-Series Split 80:20)

In [ ]:
features = [
    'Is_NFP_Week', 'Is_CPI_Day', 'XAU_DXY_Ratio_Return',
    'DXY_Return_1', 'DXY_Return_3',
    'Body_M15', 'Lower_Wick_M15', 'Upper_Wick_M15', 
    'FVG_Bull', 'FVG_Bear', 'Dist_Support', 'Dist_Resistance',
    'BOS_Bull', 'BOS_Bear', 'CHoCH_Bull', 'CHoCH_Bear',
    'Liquidity_Sweep_High', 'Liquidity_Sweep_Low', 'Order_Block_Bull',
    'Fibo_Pos_100', 'Fibo_Dist_382', 'Fibo_Dist_500', 'Fibo_Dist_618',
    'RSI_M15', 'BB_Bandwidth', 'BB_Pos',
    'XAU_Return_1', 'XAU_Return_3', 'XAU_Return_5',
    'Trend_H1_Bull', 'Trend_H1_Strong'
]

X = df_clean[features]
y = df_clean['Target_Dir']

split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"📊 Train Size (80%): {len(X_train)} samples | Test Size (20%): {len(X_test)} samples")

### Step 5: Benchmark Perbandingan Model (Bab 4 Skripsi)

Membandingkan LightGBM dengan XGBoost, Random Forest, dan Logistic Regression.

In [ ]:
models = {
    "LightGBM (Usulan)": LGBMClassifier(
        n_estimators=600, learning_rate=0.015, max_depth=6, num_leaves=25,
        min_child_samples=50, subsample=0.75, colsample_bytree=0.75,
        reg_alpha=0.1, reg_lambda=1.0, class_weight='balanced', random_state=42, verbose=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300, learning_rate=0.02, max_depth=5,
        subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='logloss'
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_split=10, random_state=42, n_jobs=-1
    ),
    "Logistic Regression": LogisticRegression(
        max_iter=1000, random_state=42
    )
}

results = []
for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    t_train = time.time() - t0
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else y_pred
    
    acc  = accuracy_score(y_test, y_pred) * 100
    prec = precision_score(y_test, y_pred) * 100
    rec  = recall_score(y_test, y_pred) * 100
    f1   = f1_score(y_test, y_pred) * 100
    auc  = roc_auc_score(y_test, y_prob) * 100
    
    results.append({
        "Model": name,
        "Akurasi (%)": round(acc, 2),
        "Precision (%)": round(prec, 2),
        "Recall (%)": round(rec, 2),
        "F1-Score (%)": round(f1, 2),
        "AUC-ROC (%)": round(auc, 2),
        "Waktu Train (s)": round(t_train, 3)
    })

df_res = pd.DataFrame(results)
display(df_res)

### Step 6: Pembuktian Win Rate dengan High-Confidence Dual Threshold & Trend Guard

In [ ]:
model_lgb = models["LightGBM (Usulan)"]
probs = model_lgb.predict_proba(X_test)
prob_up = probs[:, 1] * 100
prob_down = probs[:, 0] * 100
h1_trend_test = df_clean['Trend_H1_Bull'].iloc[split_idx:]

# 1. Tanpa Filter (Trade di Setiap Candle)
raw_winrate = (model_lgb.predict(X_test) == y_test).mean() * 100

# 2. Filter Keyakinan Model >= 60%
mask_60 = (prob_up >= 60.0) | (prob_down >= 60.0)
preds_60 = (prob_up[mask_60] >= 60.0).astype(int)
winrate_60 = (preds_60 == y_test[mask_60]).mean() * 100

# 3. Filter Keyakinan >= 60% + Trend Guard H1
mask_guard = ((prob_up >= 60.0) & (h1_trend_test == 1)) | ((prob_down >= 60.0) & (h1_trend_test == 0))
preds_guard = (prob_up[mask_guard] >= 60.0).astype(int)
winrate_guard = (preds_guard == y_test[mask_guard]).mean() * 100

print(f"❌ Win Rate Tanpa Filter (All Candles)   : {raw_winrate:.2f}%")
print(f"🟢 Win Rate dengan Filter (Prob >= 60%)  : {winrate_60:.2f}% (Total Trade: {mask_60.sum()})")
print(f"🚀 Win Rate Filter + Trend Guard H1      : {winrate_guard:.2f}% (Total Trade: {mask_guard.sum()})")

# Visualisasi Grafis Peningkatan Win Rate
plt.figure(figsize=(7, 4))
bars = plt.bar(['Tanpa Filter', 'Filter Prob >= 60%', 'Filter + Trend Guard H1'], 
               [raw_winrate, winrate_60, winrate_guard], 
               color=['#e74c3c', '#2ecc71', '#3498db'])
plt.ylabel('Win Rate (%)', fontsize=12)
plt.title('Peningkatan Win Rate Melalui Dual-Threshold Filtering', fontsize=13)
plt.ylim(40, 75)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f"{yval:.2f}%", ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()